# CDC-CP P2e: can anti-shortcut training fix unseen pair accuracy?

**The problem (from P1e).** On unseen crop-disease pairs the crop head is right about 60% of the time and the disease head
about 20%, but both are right on the same image only about 1% of the time. Without help, 99% of guesses are a pair seen
in training. The model learned whole seen pairs ("this leaf look means this crop with this disease"), not crop and disease
separately. The seen-pair penalty (calibration) lifts pair accuracy to about 29%, but only about 1 in 5 of those hits
also has the disease right: each crop has about one unseen pair, so a right crop plus the penalty already names it.
So the part to fix is the **disease on unseen pairs**.

**What this notebook tests.** Four ConvNeXt-Tiny setups, each with seeds 42, 43 and 44 (12 runs, 10 epochs each):

| Config | What it tests |
|---|---|
| `p2c_ctl` | Control: plain cross-entropy |
| `p2c_plain` | Plus contrastive loss: images with the same crop, or the same disease, pulled together |
| `p2c_cross` | Disease contrast only across crops (same disease, other crop), with a small per-pair memory |
| `p2c_cross_spy` | `p2c_cross` plus a crop "spy" trained against the disease features, so they stop carrying the crop |

Then prototype scoring (`scripts/p3_eval.py`) on the last epoch of every run.

**The verdict.** Each setup is compared with the control on unseen disease Top-1, paired by seed. It counts as better
or worse only if the gap beats a bar made of the seed spread, the noise of 400 unseen images, and a wider factor
(2.4 standard errors, not 2) because three setups are compared with one control. The output also shows the pair numbers:
both heads right, pair accuracy raw and with the penalty, and the share of penalty hits whose disease is also right.

**Smoke test first:** keep `SMOKE = True` and Run All. It trains 2 epochs of each setup (the second uses the pair memory) and prints the minutes per run.
Then set `SMOKE = False` and **Save Version, Save & Run All (Commit)**.

**Setup (once):** inputs are the code dataset (`cdc-cp_code_p2e.zip`) and the data dataset. **Accelerator -> GPU T4 x2**,
**Internet -> On**.

**If it stops early:** add this notebook's last version (its output) as an input, then Save & Run All again. Finished runs
are skipped and an interrupted run resumes.

Only `DPD_Small_train.csv` and its 8,000 images are read. The official test files are never opened.

## 1. Settings

In [ ]:
from pathlib import Path

SMOKE    = True    # True: 2 epochs of each config (seed 42) to check it runs and to time it. Then set False and commit.
TOL      = 3.0     # keep-seen: seen accuracy may fall at most this many points below raw
WORKERS  = 2       # data-loader workers per run (2 runs share Kaggle's 4 CPU cores)
Z        = 2.4     # standard errors a gap must beat; 2.4 (not 2) because 3 configs are compared with one control

CONTROL = 'p2c_ctl'
CONFIGS = [CONTROL, 'p2c_plain', 'p2c_cross', 'p2c_cross_spy']   # configs/<name>.yaml; the run folder is <name>_seed<seed>
SEEDS   = (42, 43, 44)

INPUT     = Path('/kaggle/input')
RUNS_DIR  = Path('/kaggle/working/p2e_smoke' if SMOKE else '/kaggle/working/runs')
CODE_DIR  = Path('/tmp/cdc-cp')
DATA_DIR  = Path('/tmp/DPD_Small')
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if N_GPU < 2:
    print('Only one GPU: runs will go one at a time (about twice as long). Pick GPU T4 x2 to run two at once.')

os.chdir('/kaggle/working')  # a rerun deletes CODE_DIR, so the kernel must not be inside it
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
found = next((p.parent.parent for p in INPUT.rglob('cdccp/__init__.py')), None)
if found is not None:
    shutil.copytree(found, CODE_DIR)
else:
    code_zip = next(INPUT.rglob('cdc-cp_code*.zip'), None)
    assert code_zip, 'Add the code dataset (cdc-cp_code.zip) as an input'
    with zipfile.ZipFile(code_zip) as z:
        z.extractall('/tmp')
assert (CODE_DIR / 'scripts' / 'p3_eval.py').is_file() and 'def pair_breakdown' in (CODE_DIR / 'cdccp' / 'evaluate.py').read_text(), \
    'old code dataset: upload the new cdc-cp_code_p2e.zip'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
missing_cfg = [c for c in CONFIGS if not (CODE_DIR / 'configs' / f'{c}.yaml').is_file()]
assert not missing_cfg, f'code dataset lacks configs {missing_cfg}: upload the new code zip'
from cdccp.evaluate import pair_breakdown
from cdccp.selection import compare_arms, run_scores
print('code from', found or code_zip)


def sh(*args):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore runs from earlier outputs

In [ ]:
import numpy as np
import yaml

RUNS_DIR.mkdir(parents=True, exist_ok=True)


def run_name(config, seed):
    return f'{config}_seed{seed}'


def planned_epochs(config):
    return yaml.safe_load((CODE_DIR / 'configs' / f'{config}.yaml').read_text(encoding='utf-8'))['epochs']


def epochs_done(run_dir):
    history = run_dir / 'history.csv'
    return len(pd.read_csv(history)) if history.is_file() else 0


def applied(run_dir):
    return torch.load(run_dir / 'last.pt', map_location='cpu', weights_only=False)['applied']


def finished(config, seed):
    # all planned epochs in history.csv, and last.pt and the last val_NN.npz saved after the last row
    run_dir, n = RUNS_DIR / run_name(config, seed), planned_epochs(config)
    if epochs_done(run_dir) != n or not (run_dir / 'last.pt').is_file() or not (run_dir / f'val_{n:02d}.npz').is_file():
        return False
    return applied(run_dir) == int(pd.read_csv(run_dir / 'history.csv').applied.iloc[-1])


# Copy the furthest copy of each planned run from earlier outputs. Smoke runs are never copied. The trainer itself refuses
# a run folder whose recipe differs from the config, so another config cannot be mixed in.
restored = []
if not SMOKE:
    for config in CONFIGS:
        for seed in SEEDS:
            dst = RUNS_DIR / run_name(config, seed)
            for last in INPUT.rglob(f'{run_name(config, seed)}/last.pt'):
                if 'p2e_smoke' in last.parts or 'p2_smoke' in last.parts:
                    continue
                if not (dst / 'last.pt').exists() or applied(dst) < applied(last.parent):
                    shutil.rmtree(dst, ignore_errors=True)
                    shutil.copytree(last.parent, dst)
                    restored.append(str(last.parent))
print('restored:', restored or 'nothing')

## 5. Image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Run queue
Each GPU takes the next waiting run. Only the last 3 epoch checkpoints of a run are kept.

In [ ]:
COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']


def job(config, seed=42):
    args = [sys.executable, 'scripts/train.py', f'configs/{config}.yaml', *COMMON, f'seed={seed}', '--resume']
    if SMOKE:
        args += ['--max-epochs', '2']   # epoch 2 uses the pair memory (start_epoch 1)
    return run_name(config, seed), args


def prune(run_dir, keep=3):
    # keep the last `keep` epoch checkpoints; p3_eval reads the last one
    for p in sorted(run_dir.glob('epoch_*.pt'))[:-keep]:
        p.unlink()


def run_queue(jobs):
    """Run (name, args) jobs, one per GPU at a time. A failed run is reported and left out of the ranking."""
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(RUNS_DIR / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, 0, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, seen, t0) in list(running.items()):
            log.flush()
            lines = (RUNS_DIR / f'{name}.log').read_text(errors='replace').splitlines()
            for line in lines[seen:]:
                if line.startswith(('epoch', 'done', 'nothing to do', 'Traceback')) or 'Error' in line:
                    print(f'[gpu {gpu}] {name}: {line}', flush=True)
            running[gpu] = (name, p, log, len(lines), t0)
            if p.poll() is not None:
                log.close()
                del running[gpu]
                mins = (time.time() - t0) / 60
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name} after {mins:.0f} min; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {mins:.0f} min', flush=True)
                    if (RUNS_DIR / name).is_dir():
                        prune(RUNS_DIR / name)
    if failed:
        print(f'WARNING failed runs, left out of the ranking: {failed} (logs in {RUNS_DIR})', flush=True)
    return failed


def p3_job(config, seed):
    run_dir = RUNS_DIR / run_name(config, seed)
    return f'p3_{run_name(config, seed)}', [sys.executable, 'scripts/p3_eval.py', run_dir, '--epoch', planned_epochs(config),
                                            '--tol', TOL, f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']

## 7. Train 4 configs x 3 seeds

In [ ]:
FAILED = run_queue([job(c, s) for s in ((42,) if SMOKE else SEEDS) for c in CONFIGS])
if SMOKE:
    assert not FAILED, f'smoke runs failed: {FAILED}. Read their logs in {RUNS_DIR} before the full run.'
    print('SMOKE run finished. A full run is about 5 times the minutes above (10 epochs, not 2);')
    print('12 runs go two at a time, so the whole notebook takes about 6 x 5 x those minutes, plus P3.')
    print('Set SMOKE = False and Save & Run All (Commit).')

## 8. Prototype scoring (P3) on the last epoch of every finished run

In [ ]:
if not SMOKE:
    done = [(c, s) for c in CONFIGS for s in SEEDS if finished(c, s)]
    todo = [(c, s) for c, s in done if not (RUNS_DIR / run_name(c, s) / f'p3_board_ep{planned_epochs(c):02d}.md').is_file()]
    FAILED_P3 = run_queue([p3_job(c, s) for c, s in todo])

## 9. Compare with the control
Saved as `runs/p2e_compare.md` and `runs/p2e_runs.csv`.

In [ ]:
from cdccp.catalogue import load_catalogue
from cdccp.config import load_config
from cdccp.data import load_split

if not SMOKE:
    cat = load_catalogue(CODE_DIR / load_config()['paths']['catalogue_dev'])
    rows, missing = [], []
    for c in CONFIGS:
        for s in SEEDS:
            run_dir = RUNS_DIR / run_name(c, s)
            if not finished(c, s):
                missing.append(run_name(c, s))
                continue
            h = pd.read_csv(run_dir / 'history.csv')
            sc = run_scores(h)
            with np.load(run_dir / f"val_{sc['pick_epoch']:02d}.npz") as z:
                seen = {k[len('seen_'):]: z[k] for k in z.files if k.startswith('seen_')}
                unseen = {k[len('unseen_'):]: z[k] for k in z.files if k.startswith('unseen_')}
            pb = pair_breakdown(seen, unseen, cat, tol=TOL)
            shortcut = float(h.set_index('epoch').loc[sc['pick_epoch'], 'unseen_shortcut_rate'])
            rows.append({'arm': c, 'seed': s, **sc, **pb, 'shortcut_rate': shortcut})
    run_cols = ['arm', 'seed', 'unseen_disease', 'seen_disease', 'pick_epoch', 'best_epoch', 'best_unseen_disease', 'epochs',
                'crop', 'disease', 'both_heads', 'A_S_raw', 'A_U_raw', 'gamma_keep', 'A_S_keep', 'A_U_keep', 'keep_disease_right', 'shortcut_rate']
    per_run = pd.DataFrame(rows, columns=run_cols)
    assert all(set(r) == set(run_cols) for r in rows), 'run_scores or pair_breakdown changed its keys: update run_cols'
    split_cfg = load_config(CODE_DIR / 'configs' / f'{CONTROL}.yaml', [f'paths.cache_dir={CACHE_DIR}'])
    fit, val_unseen = load_split(split_cfg, 'train_fit'), load_split(split_cfg, 'val_unseen')
    trivial = 100 * (val_unseen.disease_id == fit.disease_id.mode()[0]).mean()
    cols = ['unseen_disease', 'crop', 'both_heads', 'A_U_raw', 'A_U_keep', 'keep_disease_right', 'shortcut_rate', 'seen_disease', 'A_S_keep']
    means = per_run.groupby('arm')[cols].mean().reindex([c for c in CONFIGS if c in set(per_run.arm)]).round(1)
    lines = ['Per run. unseen_disease and seen_disease: mean of the best 3-epoch window of unseen disease Top-1, centred on',
             'pick_epoch. All other columns: the single epoch pick_epoch.',
             per_run[['arm', 'seed', 'pick_epoch', *cols]].round(1).to_string(index=False), '',
             'Mean over seeds:', means.to_string(), '',
             f'trivial: always the most common training disease scores {trivial:.2f} unseen disease Top-1', '',
             'unseen_disease: disease head right on unseen pairs (the decision column). crop: crop head right. both_heads: both right on',
             'the same image. A_U_raw / A_U_keep: unseen pair accuracy with no penalty / with the keep-seen penalty. keep_disease_right:',
             'share of A_U_keep hits whose disease head is also right. shortcut_rate: share of wrong diseases that are a disease the true',
             'crop had in training (lower is better).']
    verdicts = []
    for c in CONFIGS[1:]:
        gone = [run_name(a, s) for a in (CONTROL, c) for s in SEEDS if run_name(a, s) in missing]
        if gone:
            verdicts.append(f'{c}: NO VERDICT, runs missing or unfinished: {gone}')
            continue
        _, diffs, mean, limit, word = compare_arms(per_run, CONTROL, c, SEEDS, z=Z)
        per_seed = ', '.join(f'{s}: {d:+.1f}' for s, d in diffs.items())
        verdicts.append(f'{c}: {word.upper()}: {mean:+.1f} points of unseen disease Top-1 against {CONTROL} (bar {limit:.1f}; per seed {per_seed})')
    text = '\n'.join(lines)
    report = ('```\n' + text + '\n```\n\n' + '\n\n'.join(verdicts) + '\n\nThis compares training recipes on development validation. '
              'The score picks its epoch on the same images, so it is not a significance test and not a test-set estimate.')
    print(text, '', *verdicts, sep='\n')
    per_run.to_csv(RUNS_DIR / 'p2e_runs.csv', index=False)
    (RUNS_DIR / 'p2e_compare.md').write_text('# P2e anti-shortcut setups vs control (development validation)\n\n' + report + '\n', encoding='utf-8')
    for p in sorted(RUNS_DIR.glob('*/p3_board_ep*.md')):
        print('\n' + p.parent.name + '\n' + p.read_text(encoding='utf-8'))

## 10. Curves

In [ ]:
if not SMOKE:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    colours = dict(zip(CONFIGS, ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']))
    for c in CONFIGS:
        for s in SEEDS:
            path = RUNS_DIR / run_name(c, s) / 'history.csv'
            if not path.is_file():
                continue
            h = pd.read_csv(path)
            kw = dict(color=colours[c], marker='.', label=f'{c} s{s}')
            axes[0].plot(h.epoch, h.unseen_disease_top1, **kw)
            axes[1].plot(h.epoch, h.unseen_shortcut_rate, **kw)
            axes[2].plot(h.epoch, h.seen_disease_top1, **kw)
    titles = ['Unseen disease Top-1', 'Unseen shortcut rate (lower is better)', 'Seen disease Top-1']
    for ax, t in zip(axes, titles):
        ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.show()